In [2]:
pip install bs4


   ------------- -------------------------- 1/3 [beautifulsoup4]
   ---------------------------------------- 3/3 [bs4]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import csv
import re
import matplotlib.pyplot as plt
from urllib.parse import urljoin
import time

In [15]:
url = "https://www.keralauniversity.ac.in/affiliated-colleges"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=30)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

200


In [16]:
title = soup.find("title")

if title:
    print(title.text.strip())
else:
    print("Page title not found")

University of Kerala


In [17]:
headings = soup.find_all(["h1", "h2", "h3", "h4", "h5"])

print("Total headings:", len(headings))

for heading in headings:
    print(heading.text.strip())

Total headings: 207
University of Kerala
Affiliated Colleges
List of Affiliated Colleges in Alphabetic Order
A. J. College of Science and Technology, Thonnakkal
All Saints' College, Thiruvananthapuram
Allama Iqbal Institute of Management, Peringamala, Thiruvananthapuram
Ayyankali Memorial Arts and Science College, Kuryottumala, Vettithitta P. O., Pathanapuram
B.N.V. College of Teacher Education, Thiruvallam, Thiruvananthapuram
Baby John Memorial Govt. College, Chavara
Badhiriya B.Ed. Training College, Velichikkala, Kollam
Baselios Marthoma Mathews II Training College, Kottarakara
Bishop Jerome School of Architecture, Fathima College Road, Kollam
Bishop Jesudasan CSI Arts & Science College, Mulayara, Thiruvananthapuram
Bishop Moore College for the Hearing Impaired, Manakala, Adoor
Bishop Moore College, Mavelikkara, Alappuzha
Buddha College of Teacher Education, Cheppad, Muthukulam North, Alappuzha
C. H. M. M. College for Advanced Studies, Chavarcode, Varkala
Carmel College, Muhamma, Ala

In [18]:
links = soup.find_all("a", href=True)

print("Total links:", len(links))

for link in links[:30]:
    print(link.text.strip(), "|", link["href"])

Total links: 574
University of Kerala | index.php
Home | index.php
About
              ▾
▾ | #
General
                  ▾
▾ | #
Grievance Redressal | grievance-redressal
A-Z index | a-z-index
Instrumentation Facility (CLIF) | http://kusicc.ac.in
Site map | site-map
Overview | our-history
University Act and Statutes | acts-statutes
Institutional Development Plan | /downloads/IDP_UOK_final_version.pdf
Recognition / Approval | recognition-approval
Annual Reports | annual-reports
NAAC | /NAAC
Institutional Data (NIRF) | nirf
QS Rankings | QS-ranking
ARIIA Report | ARIIA-Report
Annual Accounts | annual-accounts
Budget Estimate | budget-estimate
Vision & Mission | vision-mission
Founder | founder
Why KU? | why-ku
Facts and Figures | facts-figures
Administration
              ▾
▾ | #
Organogram Chart | #
Chancellor | chancellor
Pro-Chancellor | pro-chancellor
Vice Chancellor | vice-chancellor
Registrar | registrar
Finance Officer | finance-officer


In [20]:
college_headings = soup.find_all(["h3", "h4"])

for heading in college_headings[:20]:
    print("College:", heading.get_text(" ", strip=True))

    link = heading.find("a", href=True)

    if link:
        print("Website:", urljoin(url, link["href"]))

    

College: University of Kerala
College: Affiliated Colleges
College: A. J. College of Science and Technology, Thonnakkal
Website: http://www.ajcollege.in
College: All Saints' College, Thiruvananthapuram
Website: http://www.allsaintscollege.ac.in
College: Allama Iqbal Institute of Management, Peringamala, Thiruvananthapuram
Website: http://www.aimmba.com
College: Ayyankali Memorial Arts and Science College, Kuryottumala, Vettithitta P. O., Pathanapuram
Website: http://www.akmascollege.com
College: B.N.V. College of Teacher Education, Thiruvallam, Thiruvananthapuram
Website: http://www.bnvtrust.org
College: Baby John Memorial Govt. College, Chavara
Website: http://www.bjmgovtcollege.ac.in
College: Badhiriya B.Ed. Training College, Velichikkala, Kollam
Website: http://www.badhiriya.org
College: Baselios Marthoma Mathews II Training College, Kottarakara
Website: http://www.bmm2tc.org
College: Bishop Jerome School of Architecture, Fathima College Road, Kollam
College: Bishop Jesudasan CSI Ar

In [21]:
college_data = []

college_headings = soup.find_all(["h3", "h4"])

for heading in college_headings:

    college_name = heading.get_text(" ", strip=True)

    if not college_name:
        continue

    college_website = ""
    email = ""
    phone = ""
    address = ""
    district = ""
    college_type = ""

    # Find the college's surrounding HTML section
    block = heading.parent

    for i in range(5):
        if block is None:
            break

        block_text = block.get_text(" ", strip=True)

        if "Estd." in block_text or "Self Financing" in block_text:
            break

        block = block.parent

    if block is None:
        block = heading.parent

    block_text = block.get_text(" ", strip=True)

    # Find links inside the college section
    block_links = block.find_all("a", href=True)

    for link in block_links:
        href = link["href"].strip()
        link_text = link.get_text(" ", strip=True)

        if href.startswith("mailto:") and not email:
            email = href.replace("mailto:", "").strip()

        elif href.startswith("tel:") and not phone:
            phone = href.replace("tel:", "").strip()

        elif href.startswith(("http://", "https://")):
            if "keralauniversity.ac.in" not in href.lower():
                if not college_website:
                    college_website = urljoin(url, href)

        elif href.startswith("www.") and not college_website:
            college_website = "https://" + href

    # Find an email address in the visible text
    if not email:
        email_match = re.search(
            r'[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}',
            block_text
        )

        if email_match:
            email = email_match.group()

    # Find a phone number in the visible text
    if not phone:
        phone_match = re.search(
            r'(?:\+91[\s-]?)?(?:0[\s-]?)?[6-9]\d{9}',
            block_text
        )

        if phone_match:
            phone = phone_match.group()

    # Extract district and college type from the listing text
    districts = [
        "Thiruvananthapuram",
        "Kollam",
        "Alappuzha",
        "Pathanamthitta"
    ]

    for d in districts:
        if d.lower() in block_text.lower():
            district = d
            break

    types = [
        "Government Self Financing",
        "Self Financing",
        "Government",
        "Aided",
        "Autonomous"
    ]

    for t in types:
        if t.lower() in block_text.lower():
            college_type = t
            break

    # Save the extracted information
    college_data.append({
        "College Name": college_name,
        "Address": address,
        "District": district,
        "College Type": college_type,
        "Contact Number": phone,
        "Email Address": email,
        "Website URL": college_website
    })

print("Records extracted:", len(college_data))

Records extracted: 206


In [22]:
df = pd.DataFrame(college_data)

df.head(10)

,College Name,Address,District,College Type,Contact Number,Email Address,Website URL
0,University of Kerala,,Thiruvananthapuram,Government Self Financing,,,http://www.ajcollege.in
1,Affiliated Colleges,,Thiruvananthapuram,Government Self Financing,,,http://www.ajcollege.in
2,"A. J. College of Science and Technology, Thonn...",,Thiruvananthapuram,Self Financing,,,http://www.ajcollege.in
3,"All Saints' College, Thiruvananthapuram",,Thiruvananthapuram,Aided,,,http://www.allsaintscollege.ac.in
4,"Allama Iqbal Institute of Management, Peringam...",,Thiruvananthapuram,Self Financing,,,http://www.aimmba.com
5,"Ayyankali Memorial Arts and Science College, K...",,Kollam,Aided,,,http://www.akmascollege.com
6,"B.N.V. College of Teacher Education, Thiruvall...",,Thiruvananthapuram,Self Financing,,,http://www.bnvtrust.org
7,"Baby John Memorial Govt. College, Chavara",,Kollam,Government,,,http://www.bjmgovtcollege.ac.in
8,"Badhiriya B.Ed. Training College, Velichikkala...",,Kollam,Self Financing,,,http://www.badhiriya.org
9,"Baselios Marthoma Mathews II Training College,...",,Kollam,Self Financing,,,http://www.bmm2tc.org


In [23]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns)

print("\nDataset information:")
df.info()

Dataset shape: (206, 7)

Column names:
Index(['College Name', 'Address', 'District', 'College Type', 'Contact Number',
       'Email Address', 'Website URL'],
      dtype='object')

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   College Name    206 non-null    object
 1   Address         206 non-null    object
 2   District        206 non-null    object
 3   College Type    206 non-null    object
 4   Contact Number  206 non-null    object
 5   Email Address   206 non-null    object
 6   Website URL     206 non-null    object
dtypes: object(7)
memory usage: 11.4+ KB


In [24]:
print("Missing values in each column:")

print(df.isnull().sum())

print("\nEmpty values in each column:")

print((df == "").sum())

Missing values in each column:
College Name      0
Address           0
District          0
College Type      0
Contact Number    0
Email Address     0
Website URL       0
dtype: int64

Empty values in each column:
College Name        0
Address           206
District            0
College Type        1
Contact Number    206
Email Address     206
Website URL        31
dtype: int64


In [25]:
print("Rows before removing duplicates:", len(df))

df = df.drop_duplicates(subset=["College Name"])

df = df.reset_index(drop=True)

print("Rows after removing duplicates:", len(df))

Rows before removing duplicates: 206
Rows after removing duplicates: 206


In [26]:
for column in df.columns:
    df[column] = df[column].fillna("").astype(str).str.strip()

df["College Name"] = df["College Name"].str.replace(
    r"\s+", " ", regex=True
)

df.head()

,College Name,Address,District,College Type,Contact Number,Email Address,Website URL
0,University of Kerala,,Thiruvananthapuram,Government Self Financing,,,http://www.ajcollege.in
1,Affiliated Colleges,,Thiruvananthapuram,Government Self Financing,,,http://www.ajcollege.in
2,"A. J. College of Science and Technology, Thonn...",,Thiruvananthapuram,Self Financing,,,http://www.ajcollege.in
3,"All Saints' College, Thiruvananthapuram",,Thiruvananthapuram,Aided,,,http://www.allsaintscollege.ac.in
4,"Allama Iqbal Institute of Management, Peringam...",,Thiruvananthapuram,Self Financing,,,http://www.aimmba.com


In [27]:
def valid_email(email):
    pattern = r'^[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}$'
    return bool(re.match(pattern, email))

df["Email Valid"] = df["Email Address"].apply(
    lambda email: valid_email(email) if email else False
)

print("Valid emails:", df["Email Valid"].sum())

print("Invalid or missing emails:", (~df["Email Valid"]).sum())

Valid emails: 0
Invalid or missing emails: 206


In [28]:
def valid_phone(phone):
    digits = re.sub(r"\D", "", phone)

    if digits.startswith("91") and len(digits) == 12:
        digits = digits[2:]

    return len(digits) == 10 and digits[0] in "6789"

df["Phone Valid"] = df["Contact Number"].apply(
    lambda phone: valid_phone(phone) if phone else False
)

print("Valid phone numbers:", df["Phone Valid"].sum())

print("Invalid or missing phone numbers:", (~df["Phone Valid"]).sum())

Valid phone numbers: 0
Invalid or missing phone numbers: 206


In [29]:
print("Total colleges:", len(df))

print(
    "Colleges with missing email:",
    (df["Email Address"] == "").sum()
)

print(
    "Colleges with missing website:",
    (df["Website URL"] == "").sum()
)

print(
    "Colleges with missing phone:",
    (df["Contact Number"] == "").sum()
)

print(
    "Unique districts:",
    df.loc[df["District"] != "", "District"].nunique()
)

Total colleges: 206
Colleges with missing email: 206
Colleges with missing website: 31
Colleges with missing phone: 206
Unique districts: 4


In [30]:
df.to_csv(
    "kerala_university_colleges.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV file saved successfully")

CSV file saved successfully
